In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import matplotlib.pyplot as plt

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
def load_tokenizer_and_model(model_id: str, device=device, dtype=None):
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model_kwargs = {}
    if dtype is not None:
        model_kwargs["torch_dtype"] = dtype
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.to(device)
    model.eval()
    return tokenizer, model

In [4]:
model_id = "/speech/dbwork/mul/spielwiese3/llm_project/llm_models/pre-trained/Meta-Llama-3-8b/"
tokenizer, model = load_tokenizer_and_model(model_id, device=device)

# sample sequence: "Hello world !"
prompt = "Dogs are considered to be our best friends."
toks = tokenizer(prompt, return_tensors="pt")
input_ids = toks.input_ids  # (1, L)
print("Initial tokens:", tokenizer.decode(input_ids[0]))
orig_ids = input_ids.clone().detach()
# flip some of the tokens
position = (3, 4)  # last token (0-based)
input_ids[0, position[0]:position[1]] = 2000#torch.randint_like(input_ids[0, position[0]:position[1]], 0, 30000)
print("Distorted tokens:", tokenizer.decode(input_ids[0]))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Initial tokens: <|begin_of_text|>Dogs are considered to be our best friends.
Distorted tokens: <|begin_of_text|>Dogsfor considered to be our best friends.


In [5]:
alpha_grid = np.logspace(-4, 0, 50)  # adjust range if needed
ground_truth_token_id = orig_ids[0, position[0]].item()

In [6]:
ground_truth_token_id, input_ids[0, position[0]].item()

(527, 2000)

In [7]:
tokenizer.decode([int(ground_truth_token_id)], skip_special_tokens=True)

' are'

In [8]:
tokenizer.decode([int(input_ids[0, position[0]].item())], skip_special_tokens=True)

'for'

In [9]:
emb_layer = model.get_input_embeddings()
emb_matrix = emb_layer.weight.detach().to(device)

# Initialize continuous token s
with torch.no_grad():
    base_embs = emb_layer(input_ids.to(device))

In [1]:
def joint_log_prob_from_inputs_embeds(model, inputs_embeds, target_ids, attention_mask=None):
    """
    Compute total log joint log P(sequence) for a causal LM using inputs_embeds.
    We compute sum_i log p(x_i | x_{<i}) for i=1..L (exclude position 0 where nothing predicts it).
    Implementation details:
      - Forward the model with inputs_embeds to obtain logits.
      - For causal LM, logits[:, :-1, :] predict tokens at positions 1..L-1 (i.e., next-token).
      - We compute the negative log-likelihood of targets shifted accordingly and sum across tokens.
    Returns: scalar log_prob (torch scalar)
    """
    # model outputs logits shape (B, L, V)
    outputs = model(inputs_embeds=inputs_embeds, return_dict=True)
    logits = outputs.logits  # (B, L, V)
    # We want p(x_i | x_<i) for i>=1. So take logits[:, :-1, :] and targets[:, 1:].
    logits_next = logits[:, :-1, :].contiguous()  # (B, L-1, V)
    target_next = target_ids[:, 1:].contiguous()  # (B, L-1)
    B, Lm1, V = logits_next.shape
    logits_flat = logits_next.view(B * Lm1, V)
    targets_flat = target_next.view(B * Lm1)
    # ignore padding if present: assume pad token id is tokenizer.pad_token_id or -100 already applied.
    # We will compute sum of negative log likelihoods for all non-ignored positions
    loss_fct = torch.nn.CrossEntropyLoss(reduction="sum", ignore_index=-100)
    nll_sum = loss_fct(logits_flat, targets_flat)  # scalar: sum of negative log-likelihoods
    log_joint = - nll_sum  # sum of log probs
    return log_joint

In [10]:
def compute_q_logprob(token_emb, theta, grad, alpha):
    """
    Computes unnormalized log q(x | theta, alpha)
    using Equation (1) from the paper.
    """
    m = theta + 0.5 * alpha * grad
    diff = token_emb - m
    dist2 = (diff * diff).sum()
    return - dist2 / (2 * alpha)

In [19]:
def run_dlp_with_gt_alpha(
        model,
        input_ids,
        orig_ids,
        position,
        steps=30,
        alpha_grid=np.logspace(-4, -1, 50),
        randomized_grad=False,
        actual_method=True,
):
    device = next(model.parameters()).device
    emb_matrix = model.get_input_embeddings().weight.detach()   # (V, D)
    vocab_size, emb_dim = emb_matrix.shape

    # Ground truth embedding
    gt_id = orig_ids[0, position[0]].item()
    emb_gt = emb_matrix[gt_id]

    if actual_method:
        distorted_id = input_ids[0, position[0]].item()
        emb_distorted = emb_matrix[distorted_id]
        theta = emb_distorted.clone().detach().requires_grad_(True)
    else:
        # Initialize continuous θ₀ (start with GT + noise)
        theta = emb_gt.clone() + 0.05 * torch.randn_like(emb_gt)
        theta = theta.detach().requires_grad_(True)

    alpha_schedule = []
    chosen_tokens = []
    theta_states = []
    all_gt_logprobs = []
    entropy_traj = []

    for t in range(steps):

        # ------------------------------------------------------------------
        # 1. Compute gradient of log prob wrt θ_t
        # ------------------------------------------------------------------
        theta_emb = theta.unsqueeze(0).unsqueeze(0)
        base = model.get_input_embeddings()(input_ids).clone()
        base[:, position[0]:position[1], :] = theta_emb
        base.requires_grad_(True)
        # input_embeds = base_embs.clone().detach()

        out = model(inputs_embeds=base)
        logits = out.logits[:, :-1]
        targets = input_ids[:, 1:]

        # Joint log-likelihood
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            reduction="sum",
        )
        # log_joint = joint_log_prob_from_inputs_embeds(
        #     model, inputs_embeds, target_ids, attention_mask
        # )
        logp = -loss
        # # random grad dirn experiment
        # grad = torch.autograd.grad(logp, base)[0][0, position[0]].detach()

        # True gradient (for norm reference)
        true_grad = torch.autograd.grad(logp, base)[0][0, position[0]].detach()

        if randomized_grad:
            # Generate random direction with same norm
            rand_dir = torch.randn_like(true_grad)
            rand_dir = rand_dir / (rand_dir.norm() + 1e-12)
            grad = rand_dir * true_grad.norm()
        else:
            grad = true_grad

        print(true_grad)
        print(grad)

        # ------------------------------------------------------------------
        # 2. Pick α* that maximizes probability of the GT token
        # ------------------------------------------------------------------
        gt_logprobs = []
        
        for alpha in alpha_grid:
            lp = compute_q_logprob(emb_gt, theta.detach(), grad, alpha)
            gt_logprobs.append(lp.item())

        assert len(gt_logprobs) == len(alpha_grid)
        
        # Normalize each step's logprob row BEFORE adding to matrix
        row = np.array(gt_logprobs.copy())
        row = (row - row.min()) / (row.max() - row.min() + 1e-12)
        all_gt_logprobs.append(row)

        gt_logprobs = np.array(gt_logprobs)
        best_idx = np.argmax(gt_logprobs)
        alpha_star = alpha_grid[best_idx]
        alpha_schedule.append(alpha_star)
        

        print(f"[Step {t}] α* = {alpha_star:.6f}")

        
        # ------------------------------------------------------------------
        # 3. Using α*, compute q for all tokens and choose next x_{t+1}
        # ------------------------------------------------------------------
        with torch.no_grad():
            # m = theta + 0.5 * alpha_star * grad
            # diffs = emb_matrix - m.unsqueeze(0)     # (V, D)
            # dist2 = (diffs * diffs).sum(dim=1)      # (V,)
            # q_vals = - dist2 / (2 * alpha_star)

            # # next line is greedy
            # next_token_id = torch.argmax(q_vals).item()

            # theta: (D,)
            # grad:  (D,)   = ∇U(θ)
            # emb_matrix: (V, D)
            # alpha_star: float
            
            # Compute differences
            diffs = emb_matrix - theta.unsqueeze(0)   # (V, D)
            
            # Term 1: 0.5 * gradᵀ (e_i - θ)
            term1 = 0.5 * (diffs @ grad)              # (V,)
            
            # Term 2: - (||e_i - θ||²) / (2α)
            dist2 = (diffs * diffs).sum(dim=1)        # (V,)
            term2 = - dist2 / (2 * alpha_star)
            
            # Final log-probabilities
            logits = term1 + term2                    # (V,)


            # Temperature scaling
            temperature = 0.3  # try 0.5, 0.3, 0.1
            
            scaled_logits = logits / temperature
            probs = torch.softmax(scaled_logits, dim=0)
            
            next_token_id = torch.multinomial(probs, num_samples=1).item()
            
            # Numerical stability trick
            # probs = torch.softmax(logits, dim=0)

            # Compute entropy
            entropy_t = -(probs * torch.log(probs + 1e-12)).sum().item()
            entropy_traj.append(entropy_t)

            
            # Sample next token
            # next_token_id = torch.multinomial(probs, num_samples=1).item()



            
            chosen_tokens.append(next_token_id)
            emb_x_next = emb_matrix[next_token_id]

            # pick the token with highest q
            next_token = tokenizer.decode([next_token_id], skip_special_tokens=True)
        
            print(f"Chosen token at step {t}: '{next_token}' | Value: {logits[next_token_id]}")
            top_n = 5
            indices = torch.argsort(logits, dim=0, descending=True)[:5]
            print(f"Entropy : {entropy_t}")
            print(f"Top 5 tokens according to the logits are: ")
            for it in indices:
                print(f"Token: {tokenizer.decode([it], skip_special_tokens=True)} | Value: {logits[it]}")

        # ------------------------------------------------------------------
        # 4. Update θ_{t+1} using DLP update rule
        # ------------------------------------------------------------------
        theta_next = 0.5 * ((theta + 0.5 * alpha_star * grad) + emb_x_next)
        theta = theta_next.detach().requires_grad_(True)

        # Store trajectory
        theta_states.append(theta.detach().cpu().numpy())
        

    return np.array(alpha_schedule), chosen_tokens, np.array(theta_states), gt_logprobs, np.array(all_gt_logprobs), entropy_traj


In [20]:
# ============================================================
# 3. Plotting helpers
# ============================================================

def plot_alpha_schedule(alpha_schedule):
    plt.figure(figsize=(7,4))
    plt.plot(alpha_schedule, marker='o')
    plt.title("Optimal α Schedule (chosen by GT probability)")
    plt.xlabel("Step")
    plt.ylabel("α*")
    plt.grid(True)
    plt.show()


def plot_heatmap(all_gt_logprobs, alpha_grid):
    all_gt_logprobs = np.array(all_gt_logprobs)

    plt.figure(figsize=(8,6))
    plt.imshow(all_gt_logprobs, aspect='auto', cmap="viridis")
    plt.colorbar(label="log q(GT | θ, α)")
    plt.xlabel("α index")
    plt.ylabel("Step")
    plt.xticks(
        ticks=[0, len(alpha_grid)//2, len(alpha_grid)-1],
        labels=[f"{alpha_grid[0]:.1e}", f"{alpha_grid[len(alpha_grid)//2]:.1e}", f"{alpha_grid[-1]:.1e}"]
    )
    plt.title("Heatmap of GT Log-Probability across α Grid")
    plt.show()



In [ ]:
# ---- Load model ----

model.to(device)
model.eval()

# ---- Input prompt ----
prompt = "Dog is considered to be our best friend."
toks = tokenizer(prompt, return_tensors="pt").to(device)
input_ids = toks.input_ids  # (1, L)
print("Initial tokens:", tokenizer.decode(input_ids[0]))
orig_ids = input_ids.clone().detach()
# flip some of the tokens
position = (3, 4)  # last token (0-based)
input_ids[0, position[0]:position[1]] = 5899#torch.randint_like(input_ids[0, position[0]:position[1]], 0, 30000)
print("Distorted tokens:", tokenizer.decode(input_ids[0]))
orig_tok = tokenizer.decode([orig_ids[0, position[0]:position[1]].item()])
print(f"Original Token which was distorted: {orig_tok}")
# Select a corrupted position (example)
# position = 3  # token index to optimize (0-based)
alpha_grid = np.logspace(-5, 1, 10000)  # adjust range if needed
# ---- Run the method ----
alpha_schedule, chosen_tokens, theta_states, heatmap_data, all_h, entropy_traj = run_dlp_with_gt_alpha(
    model=model,
    input_ids=input_ids,
    orig_ids=orig_ids,
    position=position,
    steps=50,
    alpha_grid=alpha_grid
)

# ---- Print results ----
print("\nChosen α schedule:", alpha_schedule)
print("\nChosen tokens:", chosen_tokens)

# ---- Show plots ----
# plot_alpha_schedule(alpha_schedule)
# plot_heatmap(all_h, alpha_grid)

Initial tokens: <|begin_of_text|>Dog is considered to be our best friend.
Distorted tokens: <|begin_of_text|>Dog is ten to be our best friend.
Original Token which was distorted:  considered
tensor([ 0.0086, -0.3033,  0.2331,  ..., -0.6523,  0.4169,  0.6979],
       device='cuda:0')
tensor([ 0.0086, -0.3033,  0.2331,  ..., -0.6523,  0.4169,  0.6979],
       device='cuda:0')
[Step 0] α* = 0.021102
Chosen token at step 0: ' ten' | Value: 0.0
Entropy : 2.1024523448431864e-05
Top 5 tokens according to the logits are: 
Token:  ten | Value: 0.0
Token:  Ten | Value: -5.854953765869141
Token: lıkları | Value: -5.960695743560791
Token: larından | Value: -5.961921691894531
Token: těz | Value: -5.9719414710998535
tensor([ -0.8146, -22.9394,  -4.4064,  ...,  -6.5754,   3.8713,  11.6826],
       device='cuda:0')
tensor([ -0.8146, -22.9394,  -4.4064,  ...,  -6.5754,   3.8713,  11.6826],
       device='cuda:0')
[Step 1] α* = 0.001571
Chosen token at step 1: ' ten' | Value: -29.88477325439453
Entropy 

In [ ]:
def plot_heatmap_and_schedule(all_gt_logprobs, alpha_grid, alpha_schedule):

    # Convert list[list] -> array
    gt_mat = np.array(all_gt_logprobs).T
    # shape: (len(alpha_grid), num_steps)

    num_alphas, num_steps = gt_mat.shape

    plt.figure(figsize=(10, 7))

    # Show heatmap
    img = plt.imshow(
        gt_mat,
        aspect='auto',
        cmap="viridis",
        origin="lower",
        # vmin=gt_mat.min(),
        # vmax=gt_mat.max()
        # vmin=0.0,
        # vmax=1.0     # because we normalized
    )

    plt.colorbar(label="Normalized log q(GT | θ, α)")

    plt.ylabel("α")
    plt.xlabel("Time step")

    # Fix axes so they never autoscale
    # plt.xlim(0, num_steps - 1)
    # plt.ylim(0, num_alphas - 1)

    # Y ticks as readable alpha values
    yticks = np.linspace(0, num_alphas - 1, 8, dtype=int)
    plt.yticks(yticks, [f"{alpha_grid[i]:.1e}" for i in yticks])

    # Compute α index schedule
    alpha_idx_schedule = [
        np.argmin(np.abs(alpha_grid - a)) for a in alpha_schedule
    ]

    # Make sure schedule is inside bounds
    alpha_idx_schedule = np.clip(alpha_idx_schedule, 0, num_alphas - 1)

    # Plot schedule line
    plt.plot(
        np.arange(num_steps),
        alpha_idx_schedule,
        color="white",
        linewidth=2,
        label="Optimal α*(t)"
    )

    plt.legend()
    plt.title(f"Normalized GT Log-Probability Heatmap + Optimal α Schedule\nToken: '{orig_tok}' | With Sampling")

    plt.show()


In [ ]:
plot_heatmap_and_schedule(all_h, alpha_grid, alpha_schedule)

In [ ]:
def plot_entropy(entropy_traj):
    plt.figure(figsize=(7, 4))
    plt.plot(entropy_traj, marker="o")
    plt.xlabel("Step t")
    plt.ylabel("Entropy H(q_t)")
    plt.title("Entropy of Proposal Distribution Over Time")
    plt.grid(True)
    plt.show()

In [ ]:
plot_entropy(entropy_traj)

In [ ]:
plot_entropy(entropy_traj)